In [37]:
// ==== Setup ====

import java.util.*;

// wall-clock time of r, in milliseconds
long millis(Runnable r) {
    System.gc();            // start from a clean heap: less noise
    long start = System.nanoTime();
    r.run();
    return (System.nanoTime() - start) / 1_000_000;
}

// test helpers
void check(boolean ok, String what) {
    System.out.println((ok ? "ok    " : "FAIL  ") + what);
}

# Binary trees, recap

- Each node: `data`, `left`, `right`.

In [38]:
class BTNode<E> {
    E data;
    BTNode<E> left, right;

    BTNode(E data) { this(data, null, null); }

    BTNode(E data, BTNode<E> left, BTNode<E> right) {
        this.data = data;
        this.left = left;
        this.right = right;
    }
}

In [39]:
//      A
//    /   \
//   B      C
//  / \      \
// D   E      F

// tree created bottom up
BTNode<String> d = new BTNode<>("D");
BTNode<String> e = new BTNode<>("E");
BTNode<String> b = new BTNode<>("B", d, e);
BTNode<String> f = new BTNode<>("F");
BTNode<String> c = new BTNode<>("C", null, f);
BTNode<String> root = new BTNode<>("A", b, c);

# General trees

- No bound on the number of children. The **degree** of a node: its number of children.
- A node with one fixed child field per possible child does not work: how many fields?

In [5]:
// one child reference (the first child), one sibling reference (the next one):
// together they reach every child, in order
class TreeNode<E> {
    E data;
    TreeNode<E> firstChild, nextSibling;

    TreeNode(E data) { this(data, null, null); }

    TreeNode(E data, TreeNode<E> firstChild, TreeNode<E> nextSibling) {
        this.data = data;
        this.firstChild = firstChild;
        this.nextSibling = nextSibling;
    }
}

- Children of a node, in order: `firstChild`, `firstChild.nextSibling`, `firstChild.nextSibling.nextSibling`, ...
- A node with no children: `firstChild == null`.
- The last child among siblings: `nextSibling == null`.
- A `List<TreeNode<E>>` field would also work, and is closer to what real code tends to use. This representation is taught instead because it maps *any* general, ordered tree onto a binary tree: `firstChild` plays the role of left child, `nextSibling` the role of right child. The topology does not change; it is still a general tree, only now represented with binary-tree nodes.
- Once we have `size` and the traversals (below) for binary trees, which of them can run unchanged on a general tree, through this correspondence?

# Computing the size of a tree

## Size, recursively

- Trees are recursive: remove a node's edge to its parent, and what remains is still a tree.
- `size` of an empty tree: `0`. `size` of a node: `1 + size(left) + size(right)`.

In [40]:
int size(BTNode<String> node) {
    if (node == null) return 0;
    return 1 + size(node.left) + size(node.right);
}

A real implementation keeps `size` as a field, updated on every insertion and removal. This recursive method is a template for other tree algorithms to come.

## Call trees

- A recursive call can make further recursive calls before it returns. Draw them as a tree: the first call is the root, each call it makes directly is a child, down to the base cases, which are the leaves.
- Mergesort and quicksort make up to two recursive calls each: their recursion is itself a binary tree, and its height is exactly what "how many levels" meant back then.
  - Mergesort always splits in half: height `⌊log₂ N⌋`, matching the shortest binary tree on `N` nodes.
  - Quicksort, bad pivot, sorted input: split `0` and `N - 1` every time. The recursion degenerates into a "stick": height `N - 1`, matching the tallest binary tree on `N` nodes.
- A function with a single recursive call (binary search, reversing a string): its call tree is a straight line, not really branching.
- `size(node)` makes two recursive calls per node: its call tree has the exact same shape as the tree it runs on. One call per node, one edge per child link.

In [ ]:
int sizeTraced(BTNode<String> node, int depth) {
    System.out.println("  ".repeat(depth) + (node == null ? "." : node.data));
    if (node == null) return 0;
    int sizeL = sizeTraced(node.left, depth + 1);
    int sizeR = sizeTraced(node.right, depth + 1);
    int size = 1 + sizeL + sizeR;
    return size;
}

check(sizeTraced(root, 0) == 6, "size, with a trace of every call");

A
  B
    D
      .
      .
    E
      .
      .
  C
    .
    F
      .
      .
ok    size, with a trace of every call


- The indentation above traces the call tree directly: compare it to the picture of the data tree. While `size(node)` is paused waiting for `size(node.left)`, its stack frame is still open. At any such moment, the open frames are exactly the ascendants of the node currently running.
- How many calls does `size` make in total, for a tree of `N` nodes? (Count the calls on real nodes and the calls on `null` separately.)

## Size, iteratively

- Needs an explicit stack: one entry per node whose children have not yet been counted.

In [43]:
int sizeIterative(BTNode<String> root) {
    if (root == null) return 0;
    Deque<BTNode<String>> stack = new ArrayDeque<>();
    stack.push(root);
    int count = 0;
    while (!stack.isEmpty()) {
        BTNode<String> current = stack.poll();
        count++;
        System.out.println(current.data);
        if (current.left != null) stack.offer(current.left);
        if (current.right != null) stack.offer(current.right);
    }
    System.out.println();
    return count;
}

In [44]:
check(size(root) == 6, "size, recursive");
check(sizeIterative(root) == 6, "size, iterative");

ok    size, recursive


A
B
C
D
E
F

ok    size, iterative


- An empty stack means every node has been counted.
- Replace the stack with a `Queue`: does it still count every node? What changes?

## A worked trace

Tree, written as `node(left, right)`:

`A(B(D, E(H, I)), C(F(, J), G(K, )))`

Trace of `sizeIterative`, one row per iteration:

| current | pushed | stack after |
|---|---|---|
| — | — | `[A]` |
| `A` | `B`, `C` | `[B, C]` |
| `C` | `F`, `G` | `[B, F, G]` |
| `G` | `K` | `[B, F, K]` |
| `K` | — | `[B, F]` |
| `F` | `J` | `[B, J]` |
| `J` | — | `[B]` |
| `B` | `D`, `E` | `[D, E]` |
| `E` | `H`, `I` | `[D, H, I]` |
| `I` | — | `[D, H]` |
| `H` | — | `[D]` |
| `D` | — | `[]` |

`count` reaches `11`.

In [10]:
// A(B(D, E(H, I)), C(F(, J), G(K, )))
BTNode<String> tD = new BTNode<>("D");
BTNode<String> tH = new BTNode<>("H");
BTNode<String> tI = new BTNode<>("I");
BTNode<String> tE = new BTNode<>("E", tH, tI);
BTNode<String> tB = new BTNode<>("B", tD, tE);
BTNode<String> tJ = new BTNode<>("J");
BTNode<String> tF = new BTNode<>("F", null, tJ);
BTNode<String> tK = new BTNode<>("K");
BTNode<String> tG = new BTNode<>("G", tK, null);
BTNode<String> tC = new BTNode<>("C", tF, tG);
BTNode<String> tRoot = new BTNode<>("A", tB, tC);

check(size(tRoot) == 11, "size, recursive, matches the trace");
check(sizeIterative(tRoot) == 11, "size, iterative, matches the trace");

ok    size, recursive, matches the trace
ok    size, iterative, matches the trace


# Traversing a binary tree

Visit every node exactly once, to process its data. Which order is right depends on whether a node's data is needed before its children's, after them, or in between.

Running example: an expression tree, e.g. `(A + B) * C`, where leaves are operands and internal nodes are operators. Standard orders: preorder, postorder, inorder, level order.

## Preorder

The node, then left subtree, then right subtree.

- Needed whenever a node must be processed before its children: copying a tree, printing a hierarchy (list a directory, then its contents), serializing a tree to reconstruct it top-down (exercise 7 uses exactly this property).
- On the expression tree: the operator before its operands, i.e. prefix notation.

In [45]:
void preorder(BTNode<String> node, List<String> out) {
    if (node == null) return;
    out.add(node.data);
    preorder(node.left, out);
    preorder(node.right, out);
}

## Postorder

Left subtree, then right subtree, then the node.

- Needed whenever a node depends on its children's results: `size`, height (both computed this way already), deleting a tree (free the children before the node itself, or the references needed to reach them are gone).
- On the expression tree: the operands before the operator, i.e. postfix notation. Evaluating a postfix expression with a stack is exactly this traversal.

In [12]:
void postorder(BTNode<String> node, List<String> out) {
    if (node == null) return;
    postorder(node.left, out);
    postorder(node.right, out);
    out.add(node.data);
}

## Inorder

Left subtree, then the node, then right subtree.

- On the expression tree: operand, operator, operand, the usual left-to-right (infix) reading of an expression.
- The name itself will make more sense once we see binary search trees.

In [13]:
void inorder(BTNode<String> node, List<String> out) {
    if (node == null) return;
    inorder(node.left, out);
    out.add(node.data);
    inorder(node.right, out);
}

## Level order

Level `0`, then level `1`, then level `2`, ...; each level left to right.

- Needed whenever nodes closer to the root must come first: printing a tree level by level, finding the shallowest node with some property.
- Not a recursive pattern: consecutive nodes in this order can sit in different subtrees (`E`, then `F`).

In [14]:
void levelOrder(BTNode<String> root, List<String> out) {
    if (root == null) return;
    Queue<BTNode<String>> queue = new ArrayDeque<>();
    queue.add(root);
    while (!queue.isEmpty()) {
        BTNode<String> node = queue.remove();
        out.add(node.data);
        if (node.left != null) queue.add(node.left);
        if (node.right != null) queue.add(node.right);
    }
}

In [15]:
List<String> pre = new ArrayList<>(), post = new ArrayList<>(), in = new ArrayList<>(), lvl = new ArrayList<>();
preorder(root, pre);
postorder(root, post);
inorder(root, in);
levelOrder(root, lvl);
System.out.println("preorder:    " + pre);
System.out.println("postorder:   " + post);
System.out.println("inorder:     " + in);
System.out.println("level order: " + lvl);

check(pre.equals(List.of("A", "B", "D", "E", "C", "F")), "preorder");
check(post.equals(List.of("D", "E", "B", "F", "C", "A")), "postorder");
check(in.equals(List.of("D", "B", "E", "A", "C", "F")), "inorder");
check(lvl.equals(List.of("A", "B", "C", "D", "E", "F")), "level order");

preorder:    [A, B, D, E, C, F]
postorder:   [D, E, B, F, C, A]
inorder:     [D, B, E, A, C, F]
level order: [A, B, C, D, E, F]
ok    preorder
ok    postorder
ok    inorder
ok    level order


## Depth-first and breadth-first

- Preorder, inorder, postorder: **depth-first** (DFS). One subtree is finished before the next one starts; a stack (the call stack, or an explicit one) holds the way back up.
- Level order: **breadth-first** (BFS). One level is finished before the next one starts; a queue holds the nodes waiting their turn.
- Compare `levelOrder` with `sizeIterative`: what changed?
- Both come back with graphs.

## Rebuilding a tree

- One traversal does not determine the tree: `A(B, )` and `A(, B)` have the same preorder.
- Preorder and inorder, all values distinct: they do.
  - The first value of the preorder: the root.
  - In the inorder: before the root, the left subtree; after it, the right subtree.
  - `k` values in the left subtree: the next `k` values of the preorder are its preorder. Recurse.
- `list.subList(from, to)`: a view of `list`, from `from` included to `to` excluded. No copy: `O(1)`.

In [ ]:
// all values distinct
BTNode<String> rebuild(List<String> preorder, List<String> inorder) {
    if (preorder.isEmpty()) return null;
    String rootValue = preorder.get(0);
    int k = inorder.indexOf(rootValue);             // size of the left subtree
    BTNode<String> left = rebuild(preorder.subList(1, k + 1), inorder.subList(0, k));
    BTNode<String> right = rebuild(preorder.subList(k + 1, preorder.size()),
                                   inorder.subList(k + 1, inorder.size()));
    return new BTNode<>(rootValue, left, right);
}

In [ ]:
BTNode<String> rebuilt = rebuild(pre, in);
List<String> rebuiltPost = new ArrayList<>(), rebuiltLvl = new ArrayList<>();
postorder(rebuilt, rebuiltPost);
levelOrder(rebuilt, rebuiltLvl);
check(rebuiltPost.equals(post) && rebuiltLvl.equals(lvl), "same postorder and level order as root");

- Why must the values be distinct? Two trees with preorder `A, A` and inorder `A, A`?
- Preorder and postorder: do they determine the tree?
- Cost of `rebuild`, for `N` nodes? Worst case? What would a map from each value to its position in the inorder save?

## Iterative inorder

The recursive call stack is replaced by an explicit stack: push nodes while going left; when there is nowhere further left to go, pop one, process it, then go right.

- The explicit stack plays the same role as the call stack above: at any moment, it holds exactly the ascendants of `current`.

In [16]:
List<String> inorderIterative(BTNode<String> root) {
    List<String> out = new ArrayList<>();
    Deque<BTNode<String>> stack = new ArrayDeque<>();
    BTNode<String> current = root;
    while (current != null || !stack.isEmpty()) {
        if (current != null) {
            stack.push(current);
            current = current.left;
        } else {
            current = stack.pop();
            out.add(current.data);
            current = current.right;
        }
    }
    return out;
}

In [17]:
check(inorderIterative(root).equals(in), "iterative inorder matches recursive");

ok    iterative inorder matches recursive


- Could the stack be replaced with a `Queue`? Trace a few steps: does the order still come out inorder?
- Write the iterative versions of preorder and postorder. Preorder is a small change to the code above. Postorder is not.

## Exercises

### 1. A general tree, by code
Build this org chart with `TreeNode<String>`: `CEO` has children `CTO`, `CFO`; `CTO` has children `Dev1`, `Dev2`.

- What is the degree of `CEO`? Of `CFO`?
- `TreeNode` gives the children of a node a left-to-right order. Does an org chart need one?

In [18]:
TreeNode<String> dev1 = new TreeNode<>("Dev1");
TreeNode<String> dev2 = new TreeNode<>("Dev2");
TreeNode<String> cto = new TreeNode<>("CTO", dev1, null);
dev1.nextSibling = dev2;

TreeNode<String> cfo = new TreeNode<>("CFO");
TreeNode<String> ceo = new TreeNode<>("CEO", cto, null);
cto.nextSibling = cfo;

REPL.$JShell$21$TreeNode@20263838

Degree of `CEO`: `2` (`CTO`, `CFO`). Degree of `CFO`: `0`, a leaf.

An org chart usually does not care about the order of `CTO` vs. `CFO`: the structure imposes an order that the data does not need.

### 2. Size of a general tree
Write a recursive `size` for `TreeNode<E>`. A node holds a reference to its first child only, not to each child directly.

In [19]:
int size(TreeNode<String> node) {
    if (node == null) return 0;
    int count = 1;
    for (TreeNode<String> child = node.firstChild; child != null; child = child.nextSibling)
        count += size(child);
    return count;
}

check(size(ceo) == 5, "CEO's org chart: CEO, CTO, CFO, Dev1, Dev2");

ok    CEO's org chart: CEO, CTO, CFO, Dev1, Dev2


### 3. Trace the iterative inorder
Same tree as the size trace: `A(B(D, E(H, I)), C(F(, J), G(K, )))`.

Trace `inorderIterative` on it: one row per push or pop.

| action | node | stack after |
|---|---|---|
| push | `A` | `[A]` |
| push | `B` | `[A, B]` |
| push | `D` | `[A, B, D]` |
| pop → out | `D` | `[A, B]` |
| pop → out | `B` | `[A]` |
| push | `E` | `[A, E]` |
| push | `H` | `[A, E, H]` |
| pop → out | `H` | `[A, E]` |
| pop → out | `E` | `[A]` |
| push | `I` | `[A, I]` |
| pop → out | `I` | `[A]` |
| pop → out | `A` | `[]` |
| push | `C` | `[C]` |
| push | `F` | `[C, F]` |
| pop → out | `F` | `[C]` |
| push | `J` | `[C, J]` |
| pop → out | `J` | `[C]` |
| pop → out | `C` | `[]` |
| push | `G` | `[G]` |
| push | `K` | `[G, K]` |
| pop → out | `K` | `[G]` |
| pop → out | `G` | `[]` |

Result: `D, B, H, E, I, A, F, J, C, K, G`.

In [20]:
check(inorderIterative(tRoot).equals(List.of("D", "B", "H", "E", "I", "A", "F", "J", "C", "K", "G")), "trace");

ok    trace


### 4. Iterative preorder and postorder
Write `preorderIterative` and `postorderIterative` for `BTNode<String>`.
- Preorder: a small change to `inorderIterative`.
- Postorder: harder. Hint: a traversal that visits node, then right, then left, reversed, gives postorder.

In [ ]:
List<String> preorderIterative(BTNode<String> root) {
    List<String> out = new ArrayList<>();
    if (root == null) return out;
    Deque<BTNode<String>> stack = new ArrayDeque<>();
    stack.push(root);
    while (!stack.isEmpty()) {
        BTNode<String> node = stack.pop();
        out.add(node.data);
        if (node.right != null) stack.push(node.right);   // pushed first: popped after the left child
        if (node.left != null) stack.push(node.left);
    }
    return out;
}

List<String> postorderIterative(BTNode<String> root) {
    Deque<BTNode<String>> stack = new ArrayDeque<>();
    LinkedList<String> out = new LinkedList<>();     // addFirst: builds the result in reverse
    if (root == null) return out;
    stack.push(root);
    while (!stack.isEmpty()) {
        BTNode<String> node = stack.pop();
        out.addFirst(node.data);
        if (node.left != null) stack.push(node.left);
        if (node.right != null) stack.push(node.right);
    }
    return out;
}

check(preorderIterative(tRoot).equals(List.of("A", "B", "D", "E", "H", "I", "C", "F", "J", "G", "K")), "preorder");
check(postorderIterative(tRoot).equals(List.of("D", "H", "I", "E", "B", "J", "F", "K", "G", "C", "A")), "postorder");

ok    preorder
ok    postorder


### 5. Inorder with a queue
In `inorderIterative`, replace the `Deque` used as a stack with one used as a `Queue` (`poll` instead of `pop`). Does the result still come out inorder?

No. The algorithm relies on the stack to undo exactly the descent that was just made, in reverse: the last node pushed is the first one that must come back to continue inorder. A queue returns nodes in the order they were pushed, not in reverse, so it loses track of which ancestor to return to. (This is the same reason the recursive version uses the call stack, not some other structure.)

### 6. Specify the traversals
Same tree again: `A(B(D, E(H, I)), C(F(, J), G(K, )))`.

For each of inorder, preorder, postorder, level order:
- Write out the full traversal.
- Which node is visited 5th?
- Which node is visited right after `I`?

| | traversal | 5th node | node after `I` |
|---|---|---|---|
| inorder | `D, B, H, E, I, A, F, J, C, K, G` | `I` | `A` |
| preorder | `A, B, D, E, H, I, C, F, J, G, K` | `H` | `C` |
| postorder | `D, H, I, E, B, J, F, K, G, C, A` | `B` | `E` |
| level order | `A, B, C, D, E, F, G, H, I, J, K` | `E` | `J` |

### 7. Tree reconstruction
A binary tree's preorder is `A, B, D, E, H, I, C, F, J, G, K`; its inorder is `D, B, H, E, I, A, F, J, C, K, G`. Reconstruct the tree.

- The first node of a preorder is always the root.
- Find that root in the inorder: everything before it is the left subtree, everything after is the right subtree.
- The sizes of those two parts split the rest of the preorder into the preorders of the left and right subtrees. Recurse.

Root `A`, splitting inorder into `D, B, H, E, I` (left) and `F, J, C, K, G` (right), and preorder into `B, D, E, H, I` (left) and `C, F, J, G, K` (right).

Recursing the same way on each side gives back exactly the tree from the worked trace: `A(B(D, E(H, I)), C(F(, J), G(K, )))`.